In [10]:
import torch
from torch import nn

# 输入数据
x = torch.linspace(-1, 1, 100).unsqueeze(1)

# 两个真实关系，做对比实验
targets = {
    "y = 2x + 1": 2 * x + 1,
    "y = 5x - 3": 5 * x - 3,
}

print(x.shape)

torch.Size([100, 1])


In [11]:
model = nn.Linear(1, 1)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.05
)

print(optimizer)

model = nn.Linear(1, 1)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.05
)

print(optimizer)

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.05
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.05
    maximize: False
    weight_decay: 0
)


In [12]:
def train_with_optimizer(optimizer_cls, y, lr=0.05, epochs=100):

    model = nn.Linear(1, 1)

    # 保证每个实验的初始参数相同
    with torch.no_grad():
        model.weight.fill_(0.0)
        model.bias.fill_(0.0)

    loss_fn = nn.MSELoss()

    optimizer = optimizer_cls(
        model.parameters(),
        lr=lr
    )

    losses = []

    for epoch in range(epochs):

        optimizer.zero_grad()

        y_pred = model(x)

        loss = loss_fn(y_pred, y)

        loss.backward()

        optimizer.step()

        # 记录本轮更新之后的 Loss
        with torch.no_grad():
            current_loss = loss_fn(model(x), y).item()

        losses.append(current_loss)

    return model, losses

for name, y in targets.items():

    print("=" * 30)
    print("目标:", name)
    print("=" * 30)

    sgd_model, sgd_losses = train_with_optimizer(torch.optim.SGD, y)
    adam_model, adam_losses = train_with_optimizer(torch.optim.Adam, y)

    for epoch in [1, 10, 50, 100]:

        print(f"Epoch {epoch}")

        print("SGD :", sgd_losses[epoch - 1])

        print("Adam:", adam_losses[epoch - 1])

        print()

    print("SGD")
    print("Weight:", sgd_model.weight.item())
    print("Bias:", sgd_model.bias.item())

    print("Adam")
    print("Weight:", adam_model.weight.item())
    print("Bias:", adam_model.bias.item())

    print()

目标: y = 2x + 1
Epoch 1
SGD : 2.0793259143829346
Adam: 2.195605993270874

Epoch 10
SGD : 0.8025136590003967
Adam: 1.0326602458953857

Epoch 50
SGD : 0.04278629273176193
Adam: 0.005930048413574696

Epoch 100
SGD : 0.0013441391056403518
Adam: 5.6781587773002684e-05

SGD
Weight: 1.9371305704116821
Bias: 0.9999734163284302
Adam
Weight: 2.0107152462005615
Bias: 1.00421142578125

目标: y = 5x - 3
Epoch 1
SGD : 15.223287582397461
Adam: 17.034997940063477

Epoch 10
SGD : 5.350045680999756
Adam: 13.157126426696777

Epoch 50
SGD : 0.2674872875213623
Adam: 3.1757614612579346

Epoch 100
SGD : 0.008400877937674522
Adam: 0.43148502707481384

SGD
Weight: 4.8428263664245605
Bias: -2.99992036819458
Adam
Weight: 3.8777692317962646
Bias: -2.9434001445770264

